In [1]:
# Imports
from tqdm import tqdm
import os
import csv
from results.code_gen.code_gen_util import *
from ml4setk.Parsing.Code.TreeSitterQuery import TreeSitterQuery
import sys
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import StoppingCriteriaList
import torch
import math

In [2]:
language = "java"
tree_query = TreeSitterQuery(language=language)
base_dir = "../../../data"

amount_comment = 2000
amount_method = 500
smell_index = -1  # Default to -1 for multiple out-smell cases
distances = [0, 1, -1]  # Distances for out-smell
version = "v2-ds-all-1"  # Version of the input-target dataset
output_dir_input_target = f"{base_dir}/input-target/{version}"
cases = [
    f"in_smell_{amount_comment}",
    f"no_smell_comment_{amount_comment}",
    f"out_smell_distance(0)_{amount_method}",
    f"out_smell_distance(1)_{amount_method}",
    f"out_smell_distance(-1)_{amount_method}",
    f"no_smell_method_{amount_method}",
    f"multi({smell_index})_out_smell_distance({distances[0]})_{amount_method}",
    f"multi({smell_index})_out_smell_distance({distances[-1]})_{amount_method}",
    f"preprocessed_multi({smell_index})_out_smell_distance({distances[0]})_{amount_method}",
    f"preprocessed_multi({smell_index})_out_smell_distance({distances[-1]})_{amount_method}",
]

c:\Users\lucaw\TUProjects\RP\ML4SE-toolkit\.venv\Lib\site-packages\tree_sitter\__init__.py:36: FutureWarning: Language(path, name) is deprecated. Use Language(ptr, name) instead.
  warn("{} is deprecated. Use {} instead.".format(old, new), FutureWarning)


In [3]:
model_name = "SmolLM135M"
# model_name = "StarCoder2_3B"
# model_name = "MellumBase4B"

print(f"Using model: {model_name}")
checkpoint = "HuggingFaceTB/SmolLM2-135M"
# checkpoint = "bigcode/starcoder2-3b"
# checkpoint = "JetBrains/Mellum-4b-base"
device = "cuda" # cuda for GPU usage or "cpu" for CPU usage
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# for multiple GPUs install accelerate and do `model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto")`
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

Using model: SmolLM135M


In [4]:
max_tokens_comment = math.ceil(load_threshold(f"comment_stats_all_{model_name}.json")["all"]["threshold"])
max_tokens_method = math.ceil(load_threshold(f"method_stats_all_{model_name}.json")["method"]["threshold"])
max_context = model.config.max_position_embeddings
if model_name == "SmolLM135M":
    max_context = 2048
    max_tokens_method = math.ceil(load_threshold(f"method_stats_all_{model_name}.json")["method"]["mean"] + load_threshold(f"method_stats_all_{model_name}.json")["method"]["std"])
print(f"Max context size: {max_context}")
available_context_comment = max_context - max_tokens_comment
available_context_method = max_context - max_tokens_method

Max context size: 2048


In [5]:
def generate_completion_comment(input: str, comment_syntax: str, filename: str) -> str:
    inputs = tokenizer(input, return_tensors="pt", truncation=False).to(model.device)
    # Ensure the input does not exceed the available context
    if inputs["input_ids"].shape[1] > available_context_comment:
        inputs["input_ids"] = inputs["input_ids"][:, -available_context_comment:]
        print(f"Input truncated to fit within available context, file: {filename}")

    # Calculate where the generation starts (token-wise)
    start_len = inputs["input_ids"].shape[1]

    stopping_criteria = StoppingCriteriaList()

    # Choose appropriate stop tokens
    if comment_syntax.startswith("//"):
        stop_strings = ["\n"]
        stopping_criteria.append(StopOnSubstrings(stop_strings, tokenizer, start_len))
        stopping_criteria.append(RepetitionInSingleLineComment(tokenizer, start_len, filename))
    elif comment_syntax.startswith("/*") or comment_syntax.startswith("/**"):
        stop_strings = ["*/"]
        stopping_criteria.append(StopOnSubstrings(stop_strings, tokenizer, start_len))
        stopping_criteria.append(LineRepetitionStoppingCriteria(tokenizer, start_len, filename))

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_tokens_comment,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria
        )

    generated_ids = outputs[0][start_len:]
    if generated_ids[-1].item() == tokenizer.eos_token_id:
        print("Last token is EOS, the generation was stopped by the model for file:", filename)
    completion = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return completion

In [6]:
def generate_completion_method(input: str, filename: str) -> str:
    inputs = tokenizer(input, return_tensors="pt").to(model.device)
    # Ensure the input does not exceed the available context
    if inputs["input_ids"].shape[1] > available_context_method:
        # Truncate the input to fit within the available context
        inputs["input_ids"] = inputs["input_ids"][:, -available_context_method:]
        print(f"Input truncated to fit within available context, file: {filename}")

    # Calculate where the generation starts (token-wise)
    start_len = inputs["input_ids"].shape[1]
    
    stopping_criteria = StoppingCriteriaList([
        StopOnMethodTreeSitter(tree_query, tokenizer, start_len),
        LineRepetitionStoppingCriteria(tokenizer, start_len, filename),
        RepetitionInLongSingleLine(tokenizer, start_len, filename)
    ])

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_tokens_method,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria
        )

    generated_ids = outputs[0][start_len:]
    # Check if last token is end of text token
    if generated_ids[-1].item() == tokenizer.eos_token_id:
        print("Last token is EOS, the generation was stopped by the model for file:", filename)
    completion = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    methods = tree_query.parse(completion, '(method_declaration) @method')
    if methods:
        method_code = methods[0][2].strip()
        return method_code
    else:
        return completion

In [7]:
output_dir_gen = f"{base_dir}/results/{model_name}/{version}/test"
if not os.path.exists(output_dir_gen):
        os.makedirs(output_dir_gen)

In [8]:
for case in cases[0:1]:  # Process only the first two cases for comments
    samples = load_jsonl_gz(os.path.join(output_dir_input_target, f"{case}.jsonl.gz"))
    output_file_gen = f"{case}-{model_name}.csv"
    with open(os.path.join(output_dir_gen, output_file_gen), mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["filename", "target", "prediction"])

        for sample in tqdm(samples[:30], desc=f"Processing {case} samples"):
            filename = sample["file_name"]
            prefix = sample["prefix"]
            target = sample["target"]
            comment_syntax = sample["comment_syntax"]
            
            try:
                # Generate the completion
                pred = generate_completion_comment(prefix, comment_syntax, filename)
            except Exception as e:
                print(f"Error generating completion for {filename}: {e}")
                continue
            writer.writerow([filename, target, pred])
            file.flush()
            os.fsync(file.fileno())
        

Processing in_smell_2000 samples:  10%|█         | 3/30 [00:02<00:18,  1.43it/s]

Input truncated to fit within available context, file: ImageXmlFactory.java


Processing in_smell_2000 samples:  17%|█▋        | 5/30 [00:20<02:01,  4.86s/it]

Input truncated to fit within available context, file: PhoneCallReceiver.java


Processing in_smell_2000 samples:  20%|██        | 6/30 [00:20<01:20,  3.34s/it]

...

Processing in_smell_2000 samples:  90%|█████████ | 27/30 [00:37<00:02,  1.33it/s]

Input truncated to fit within available context, file: NovelReaderActivity.java


Processing in_smell_2000 samples:  97%|█████████▋| 29/30 [00:37<00:00,  1.73it/s]

Input truncated to fit within available context, file: NetworkHooks.java


Processing in_smell_2000 samples: 100%|██████████| 30/30 [00:38<00:00,  1.27s/it]


In [9]:
for case in cases[9:]: # Skip the first two cases which are for comments
    samples = load_jsonl_gz(os.path.join(output_dir_input_target, f"{case}.jsonl.gz"))
    output_file_gen = f"{case}-{model_name}-2.csv"
    with open(os.path.join(output_dir_gen, output_file_gen), mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["filename", "target", "prediction"])

        for sample in tqdm(samples[65:66], desc=f"Processing {case} samples"):
            filename = sample["file_name"]
            print(f"Processing file: {filename}")
            prefix = sample["prefix"]
            target = sample["target"]
            
            try:
                # Generate the completion
                pred = generate_completion_method(prefix, filename)
            except Exception as e:
                print(f"Error generating completion for {filename}: {e}")
                continue
            writer.writerow([filename, target, pred])
            file.flush()
            os.fsync(file.fileno())

Processing preprocessed_multi(-1)_out_smell_distance(-1)_500 samples:   0%|          | 0/1 [00:00<?, ?it/s]

Processing file: WorldGenSmallPigmanVillage.java
Input truncated to fit within available context, file: WorldGenSmallPigmanVillage.java


Processing preprocessed_multi(-1)_out_smell_distance(-1)_500 samples: 100%|██████████| 1/1 [00:08<00:00,  8.21s/it]

Stopping in file:WorldGenSmallPigmanVillage.java due to excessive whitespace


In [10]:
def generate_fim_completion_comment(prefix: str, suffix: str, comment_syntax: str, filename: str) -> str:
    fim_input_ids = get_fim_input_ids(tokenizer, prefix, suffix, available_context_comment).to(model.device)
    attention_mask = torch.ones_like(fim_input_ids)
    inputs = {
        "input_ids": fim_input_ids,
        "attention_mask": attention_mask
    }
    
    start_len = fim_input_ids.shape[1]

    stopping_criteria = StoppingCriteriaList()

    # Choose appropriate stop tokens
    if comment_syntax.startswith("//"):
        stop_strings = ["\n"]
        stopping_criteria.append(StopOnSubstrings(stop_strings, tokenizer, start_len))
        stopping_criteria.append(RepetitionInSingleLineComment(tokenizer, start_len, filename))
    elif comment_syntax.startswith("/*") or comment_syntax.startswith("/**"):
        stop_strings = ["*/"]
        stopping_criteria.append(StopOnSubstrings(stop_strings, tokenizer, start_len))
        stopping_criteria.append(LineRepetitionStoppingCriteria(tokenizer, start_len, filename))

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_tokens_comment,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria
        )

    generated_ids = outputs[0][start_len:]
    completion = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return completion

In [12]:
output_dir_gen = f"{base_dir}/results/{model_name}/{version}/fim"
if not os.path.exists(output_dir_gen):
        os.makedirs(output_dir_gen)

In [16]:
for case in cases[0:1]:
    samples = load_jsonl_gz(os.path.join(output_dir_input_target, f"{case}.jsonl.gz"))
    output_file_gen = f"fim-{case}-{model_name}.csv"
    with open(os.path.join(output_dir_gen, output_file_gen), mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["filename", "target", "prediction"])

        for sample in tqdm(samples[:3], desc=f"Processing {case} samples"):
            filename = sample["file_name"]
            prefix = sample["prefix"]
            target = sample["target"]
            suffix = sample["suffix"]
            comment_syntax = sample["comment_syntax"]
            
            try:
                # Generate the completion
                pred = generate_fim_completion_comment(prefix, suffix, comment_syntax, filename)
            except Exception as e:
                print(f"Error generating completion for {filename}: {e}")
                continue
            writer.writerow([filename, target, pred])
            file.flush()
            os.fsync(file.fileno())

Processing in_smell_2000 samples: 100%|██████████| 3/3 [06:29<00:00, 130.00s/it]
